# PASSO 1: Notebook de limpeza e padronização de dados vindos do BIM

Este notebook define e executa a função padronizar_bim_multi_casa(...) que:

- Lê o dicionário de insumos e todas as abas dos arquivos de materiais (casas_materiais.xlsx) e ambientes (casas_ambientes.xlsx, onde cada aba = uma moradia).
- Normaliza nomes/colunas e converte unidades (área/volume) para valores numéricos.
- Cria chaves normalizadas e faz merge com o dicionário para mapear insumos e elementos.
- Ajusta Pavimentos quando padrões específicos são detectados e define a unidade_padrao (m²/m³).
- Registra materiais sem correspondência, duplicatas no dicionário e atualiza o dicionário com novos itens marcados como 'VERIFICAR'.
- Gera/atualiza um arquivo consolidado (Materiais_Padronizados, Ambientes_Padronizados) e um arquivo de verificação com abas de diagnóstico.

Saída: retorna os DataFrames consolidados (df_materiais_final, df_ambientes_final) e escreve arquivos Excel de saída e verificação.


Notas importantes:
- Verifique presença da aba dicionario_insumos e das colunas obrigatórias no dicionário antes de executar.
- O código já inclui mensagens de diagnóstico e blocos de proteção para index duplicado; revisar avisos gerados no arquivo de verificação.
- Caso a execução retorne a mensagem "🔍 Linhas sem correspondência (nesta execução):" com valor diferente de zero, atualize os campos indicados no arquivo dicionario_insumos.xlsx com a mensagem 'VERIFICAR'. Use a aba SEM_CORRESPONDENCIA do arquivo verificacao_ultima_execucao.xlsx para identificar a origem dos materiais não encontrados e listados no dicionário. Após o preenchimento e correções **execute novamente o código** para gerar o arquivo consolidado final.


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import List, Dict, Any

# --- Funções de Limpeza ---
def clean(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.select_dtypes(include="object"):
        df[c] = df[c].astype(str).str.strip()
    return df

def to_float_strip_units(s: pd.Series) -> pd.Series:
    cleaned = (
        s.astype(str)
         .str.lower()
         .str.replace(",", ".", regex=False)
         .str.replace(r"[^0-9.\-e]", "", regex=True)
         .replace("", np.nan)
    )
    return pd.to_numeric(cleaned, errors="coerce")

def key(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().str.lower()


# --- Função Principal ---

def padronizar_bim_multi_casa(
    path_materiais: str,
    path_ambientes: str,
    path_dicionario: str,
    path_saida: str,
    path_verificacao: str,
    sheet_dic: str = "dicionario_insumos",
    header_row_dic: int = 1
):
    
    print("Iniciando processo de padronização...")
    path_dic = Path(path_dicionario)
    path_out = Path(path_saida)
    path_verif = Path(path_verificacao)

    # === 1. Ler Dicionário ===
    try:
        dic_raw = pd.read_excel(path_dic, sheet_name=sheet_dic, header=header_row_dic)
        dic = clean(dic_raw.copy())
    except FileNotFoundError:
        print(f"ERRO: Arquivo de dicionário não encontrado em {path_dic}")
        return
    except ValueError as e:
        print(f"ERRO: Aba '{sheet_dic}' não encontrada em {path_dic}. Detalhe: {e}")
        return

    # === 2. Ler Múltiplas Abas ===
    
    def ler_abas_por_casa(xls_path: str, tipo_dado: str) -> pd.DataFrame:
        try:
            xls = pd.ExcelFile(xls_path)
        except FileNotFoundError:
            print(f"AVISO: Arquivo de {tipo_dado} não encontrado em {xls_path}. Pulando...")
            return pd.DataFrame() 
            
        lista_dfs = []
        sheet_names = xls.sheet_names
        
        if not sheet_names:
            print(f"AVISO: Nenhuma aba encontrada em {xls_path} para {tipo_dado}.")
            return pd.DataFrame()

        print(f"Lendo {len(sheet_names)} casas de {tipo_dado} de {xls_path}...")
        for id_moradia in sheet_names:
            try:
                df = pd.read_excel(xls, sheet_name=id_moradia)
                
                if tipo_dado == "materiais":
                    rename_map = {
                        'Categoria': 'categoria_bim',
                        'Material: Nome': 'material_bim',
                        'Material: Área': 'area_bim',
                        'Material: Volume': 'volume_bim'
                    }
                    df = df.rename(columns=lambda c: rename_map.get(str(c).strip(), c), inplace=False)

                    if 'Contagem' in df.columns:
                        df = df.dropna(subset=['Contagem'])
                    else:
                        print(f"  - AVISO [{id_moradia}]: Coluna 'Contagem' não encontrada para filtro.")

                    col_pav_nome = 'PAVIMENTO'
                    col_niv_nome = 'Nível'
                    
                    col_pav = df[col_pav_nome].copy() if col_pav_nome in df.columns else None
                    col_niv = df[col_niv_nome].copy() if col_niv_nome in df.columns else None
                    
                    nova_col_pav = pd.Series(index=df.index, dtype=object) 

                    if col_pav is not None:
                        nova_col_pav = col_pav 
                    
                    if col_niv is not None:
                        nova_col_pav = nova_col_pav.fillna(col_niv)
                    
                    df['pavimento_bim'] = nova_col_pav 
                    
                    cols_to_drop = [col_pav_nome, col_niv_nome, 'Contagem']
                    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

                df["id_moradia"] = id_moradia 
                lista_dfs.append(df)
            except Exception as e:
                print(f"  - Erro ao ler aba '{id_moradia}' de {tipo_dado}: {e}")
                
        if not lista_dfs:
            return pd.DataFrame()
            
        return clean(pd.concat(lista_dfs, ignore_index=True))

    dados_raw = ler_abas_por_casa(path_materiais, "materiais")
    amb_raw = ler_abas_por_casa(path_ambientes, "ambientes")

    casas_nesta_execucao = set(dados_raw["id_moradia"].unique()).union(set(amb_raw["id_moradia"].unique()))
    if not casas_nesta_execucao:
        print("Nenhum dado de material ou ambiente foi carregado. Encerrando.")
        return

    print(f"Casas sendo processadas: {casas_nesta_execucao}")

# === 3. Processamento de Materiais ===
    
    # --- Validações e Helpers ---
    if not dados_raw.empty:
        for c in ["material_bim", "pavimento_bim"]: 
            if c not in dados_raw.columns:
                print(f"ERRO: Coluna '{c}' faltando nos dados de materiais. Verifique as abas.")
                return
    if not dic.empty:
        for c in ["nome_bim", "material_bim", "elemento", "categoria", "insumo"]:
            if c not in dic.columns:
                print(f"ERRO: Coluna '{c}' faltando no dicionário. Verifique o arquivo.")
                return

    def limpar_area_volume(df):
        for col in ["area_bim", "volume_bim"]:
            if col in df.columns:
                df[col] = to_float_strip_units(df[col])
        return df

    def pav_to_idx(pav: str) -> int:
        s = str(pav).strip().upper()
        if s == "TÉRREO" or s == "TERREO": return 0
        m = re.search(r"PAVIMENTO\s*(\d+)", s)
        return int(m.group(1)) if m else 0

    def idx_to_pav(idx: int) -> str:
        return "TÉRREO" if idx <= 0 else f"PAVIMENTO {idx}"

    SHIFT_DOWN_PATTERNS = [
        r"^\*?\s*ESTRUTURA\s+CONCRETO\s+VIGAS\s*$",
        r"^TERÇA\s*-\s*MADEIRA\s*$",
        r"^\.?\s*TELHA\s+FIBROCIMENTO\s*$",
    ]
    shift_down_regex = re.compile("|".join(SHIFT_DOWN_PATTERNS), flags=re.IGNORECASE)
    # --- Fim dos Helpers ---

    if not dados_raw.empty:
        dados = limpar_area_volume(dados_raw.copy())
        
        if "categoria_bim" not in dados.columns:
            dados["categoria_bim"] = np.nan 

        dados["_key_mat"] = key(dados["material_bim"])
        dic["_key_mat"] = key(dic["nome_bim"])
        has_un_mult = "unidade_multiplicadora" in dic.columns

        # --- Diagnóstico de Duplicatas no Dicionário ---
        dic_dups_check = dic[dic.duplicated('_key_mat', keep=False)]
        if not dic_dups_check.empty:
            print("\n--- DEBUG: AVISO! Chaves duplicadas encontradas no Dicionário ANTES do drop ---")
            print(dic_dups_check[['_key_mat', 'nome_bim', 'insumo']].sort_values('_key_mat'))
            print("------------------------------------------------------------------------------\n")
        
        print(f"DEBUG 1: O índice de 'dados' (antes do merge) é único? {dados.index.is_unique}")

        merged = dados.merge(
            dic.drop_duplicates(subset=["_key_mat"]), 
            on=["_key_mat"],
            how="left",
            suffixes=("_dados", "_dic") 
        )

        # === Ajuste de pavimento ===
        out = merged.copy()
        
        print(f"DEBUG 2: O índice de 'out' (APÓS o merge) é único? {out.index.is_unique}")
        
        out = out.reset_index(drop=True)
        
        print(f"DEBUG 3: O índice de 'out' (APÓS o reset_index) é único? {out.index.is_unique}")

        
        out["_pav_idx"] = out["pavimento_bim"].apply(pav_to_idx)
        moradias_com_superior = set(
            out.groupby("id_moradia")["_pav_idx"].max().pipe(lambda s: s[s > 0].index)
        )
        
        mask_shift = (
            out["material_bim_dados"].astype(str).str.match(shift_down_regex, na=False)
            & out["id_moradia"].isin(moradias_com_superior)
            & (out["_pav_idx"] > 0)
        )
        
        print("DEBUG 4: Prester a executar o '.loc' que pode causar erro...")
        
        # --- BLOCO TRY...EXCEPT PARA DIAGNÓSTICO DE EXECUÇÃO --- 
        try:
            # Tenta a atribuição
            out.loc[mask_shift, "_pav_idx"] = out.loc[mask_shift, "_pav_idx"] - 1 
        except ValueError as e:
            print("\n--- ERRO CAPTURADO: O 'out.loc' FALHOU ---")
            print(f"Mensagem: {e}")
            print("Isso confirma que o índice 'out.index' tem duplicatas.")
            print(f"O índice 'out' é único? {out.index.is_unique}")
            
            # Mostra as duplicatas
            if not out.index.is_unique:
                print("--- MOSTRANDO AS LINHAS COM ÍNDICE DUPLICADO ---")
                dups = out.index[out.index.duplicated()].unique()
                print(out[out.index.isin(dups)].sort_index())
            print("--------------------------------------------------\n")
            # Re-lança o erro para parar o script
            raise e
        # --- FIM DO BLOCO DE DIAGNÓSTICO --- 
            
        print("DEBUG 5: '.loc' executado com sucesso.")
        
        out["pavimento_bim"] = out["_pav_idx"].apply(idx_to_pav)
        out.drop(columns=["_pav_idx"], inplace=True)

        # === Unidade_padrao ===
        for col in ["Contagem", "area_bim", "volume_bim"]:
            if col not in out.columns: out[col] = np.nan

        if has_un_mult:
            conds = [
                out["unidade_multiplicadora"].str.contains("m2", na=False, case=False).values,
                out["unidade_multiplicadora"].str.contains("m3", na=False, case=False).values,
            ]
            choices = [
                out["area_bim"].values,
                out["volume_bim"].values,
            ]
            
            print("DEBUG 6: Prester a executar o 'np.select' que causa o erro...")
            out["unidade_padrao"] = np.select(conds, choices, default=np.nan) 
            print("DEBUG 7: 'np.select' executado com sucesso.")
            
        else:
            out["unidade_padrao"] = np.nan

        # === Selecionar colunas de saída ===
        out.rename(columns={
            "categoria_bim": "categoria", 
            "elemento_dic": "elemento",
            "categoria_dic": "categoria_dicionario", 
            "insumo_dic": "insumo",
            "material_bim_dados": "material_bim" 
        }, inplace=True)
        
        cols = [
            "id_moradia", "pavimento_bim", "material_bim",
            "categoria", "elemento", "insumo",
            "unidade_padrao", "area_bim", "volume_bim",
        ]
        for c in cols:
            if c not in out.columns:
                out[c] = np.nan
                
        padronizado = out[cols].copy()
        
        sem_corr = out[out["insumo"].isna()].copy()
        dic_dups = (
            dic.groupby(["_key_mat"], as_index=False)
            .size().query("size > 1")
        )
    
    else: 
        padronizado = pd.DataFrame(columns=[
            "id_moradia", "pavimento_bim", "material_bim",
            "categoria", "elemento", "insumo",
            "unidade_padrao", "area_bim", "volume_bim",
        ])
        sem_corr = pd.DataFrame()
        dic_dups = pd.DataFrame()
        print("AVISO: Nenhum dado de material processado.")

    # === 4. Processar aba de AMBIENTES ===
    if not amb_raw.empty:
        amb = amb_raw.copy()
        
        for col in ["area_bim", "volume_bim", "voluma_bim"]:
            if col in amb.columns:
                amb[col] = to_float_strip_units(amb[col])

        perim_cols = [c for c in amb.columns if c.lower() in ("perimetro", "perímetro")]
        for col in ["pe_direito"] + perim_cols:
            if col in amb.columns:
                amb[col] = to_float_strip_units(amb[col]) / 100.0

        ambientes_pad = amb.copy()
    else:
        ambientes_pad = pd.DataFrame() 
        print("AVISO: Nenhum dado de ambiente processado.")


    # === 5. Atualizar Dicionário ===
    if not sem_corr.empty:
        dic_keys_set = set(dic["_key_mat"])
        
        materiais_sem_corr = sem_corr[["material_bim", "_key_mat"]].drop_duplicates()
        
        novos_para_dic = materiais_sem_corr[
            ~materiais_sem_corr["_key_mat"].isin(dic_keys_set)
        ]

        if not novos_para_dic.empty:
            print(f"ATUALIZANDO DICIONÁRIO: {len(novos_para_dic)} novos materiais encontrados.")
            
            df_novos = pd.DataFrame()
            df_novos["nome_bim"] = novos_para_dic["material_bim"] 
            df_novos["material_bim"] = novos_para_dic["material_bim"] 
            df_novos["elemento"] = "VERIFICAR" # Placeholder para usuario preencher após execução com base na convenção adotada
            df_novos["categoria"] = "VERIFICAR" # Placeholder para usuario preencher após execução com base na convenção adotada
            df_novos["insumo"] = "VERIFICAR" # Placeholder para usuario preencher após execução com base na convenção adotada

            dic_atualizado = pd.concat([dic_raw, df_novos], ignore_index=True)
            
            try:
                with pd.ExcelWriter(path_dic, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
                    dic_atualizado.to_excel(writer, sheet_name=sheet_dic, index=False)
                print(f"✅ Dicionário salvo em: {path_dic} (Aba: {sheet_dic})")
            except Exception as e:
                print(f"ERRO AO SALVAR DICIONÁRIO: {e}")
                print(f"  - Verifique se o arquivo {path_dic} não está aberto.")


    # === 6. Salvar Saída Consolidada ===
    try:
        df_materiais_antigo = pd.read_excel(path_out, sheet_name="Materiais_Padronizados")
        df_ambientes_antigo = pd.read_excel(path_out, sheet_name="Ambientes_Padronizados")
        
        df_materiais_antigo = df_materiais_antigo[
            ~df_materiais_antigo['id_moradia'].isin(casas_nesta_execucao)
        ]
        df_ambientes_antigo = df_ambientes_antigo[
            ~df_ambientes_antigo['id_moradia'].isin(casas_nesta_execucao)
        ]
        
    except FileNotFoundError:
        print(f"Arquivo de saída {path_out} não encontrado. Criando um novo.")
        df_materiais_antigo = pd.DataFrame()
        df_ambientes_antigo = pd.DataFrame()
    except ValueError as e: 
        print(f"AVISO: {e}. Recriando aba(s) em {path_out}.")
        df_materiais_antigo = pd.DataFrame()
        df_ambientes_antigo = pd.DataFrame()

    df_materiais_final = pd.concat([df_materiais_antigo, padronizado], ignore_index=True)
    df_ambientes_final = pd.concat([df_ambientes_antigo, ambientes_pad], ignore_index=True)

    try:
        with pd.ExcelWriter(path_out, engine="openpyxl") as wr:
            df_materiais_final.to_excel(wr, sheet_name="Materiais_Padronizados", index=False)
            df_ambientes_final.to_excel(wr, sheet_name="Ambientes_Padronizados", index=False)
        print(f"✅ Arquivo CONSOLIDADO salvo em: {path_out}")
    except Exception as e:
        print(f"ERRO AO SALVAR SAÍDA CONSOLIDADA: {e}")
        print(f"  - Verifique se o arquivo {path_out} não está aberto.")


    # === 7. Salvar Excel de Verificação ===
    try:
        with pd.ExcelWriter(path_verif, engine="openpyxl") as wr:
            padronizado.to_excel(wr, sheet_name="PADRONIZADO_RUN", index=False)
            if ambientes_pad is not None:
                ambientes_pad.to_excel(wr, sheet_name="DIM_AMBIENTES_RUN", index=False)
            sem_corr.to_excel(wr, sheet_name="SEM_CORRESPONDENCIA", index=False)
            dic_dups.to_excel(wr, sheet_name="DIC_DUPLICADOS", index=False)
            dados_raw.to_excel(wr, sheet_name="RAW_MATERIAIS", index=False)
            amb_raw.to_excel(wr, sheet_name="RAW_AMBIENTES", index=False)
            dic_raw.to_excel(wr, sheet_name="RAW_DIC", index=False)
        print(f"✅ Arquivo de VERIFICAÇÃO salvo em: {path_verif}")
    except Exception as e:
        print(f"ERRO AO SALVAR ARQUIVO DE VERIFICAÇÃO: {e}")
        print(f"  - Verifique se o arquivo {path_verif} não está aberto.")

    print(f"🔍 Linhas sem correspondência (nesta execução): {len(sem_corr)}")
    print("Processo concluído.")
    
    return df_materiais_final, df_ambientes_final

# Aplicação de limpeza e padronização de dados vindos do BIM

In [2]:
# --- Caminhos dos arquivos ---
# (Substitua pelos seus caminhos reais)

# 1. ENTRADA
p_materiais = r"/Users/camiladuelisviana/Desktop/MORE/Integracao/casas_materiais.xlsx"
p_ambientes = r"/Users/camiladuelisviana/Desktop/MORE/Integracao/casas_ambientes.xlsx"
p_dicionario = r"/Users/camiladuelisviana/Desktop/MORE/Integracao/dicionario_insumos.xlsx"

# 2. SAÍDA
p_saida_final = r"/Users/camiladuelisviana/Desktop/MORE/Integracao/consolidado_geral.xlsx"
p_verificacao_run = r"/Users/camiladuelisviana/Desktop/MORE/Integracao/verificacao_ultima_execucao.xlsx"

# --- Executar a função ---
try:
    padronizar_bim_multi_casa(
        path_materiais=p_materiais,
        path_ambientes=p_ambientes,
        path_dicionario=p_dicionario,
        path_saida=p_saida_final,
        path_verificacao=p_verificacao_run,
        sheet_dic="dicionario_insumos", # Nome da aba no p_dicionario
        header_row_dic=0 # O header está na linha 1 (índice 0)
    )
except Exception as e:
    print(f"Ocorreu um erro inesperado: {e}")

Iniciando processo de padronização...
Lendo 39 casas de materiais de /Users/camiladuelisviana/Desktop/MORE/Integracao/casas_materiais.xlsx...
Lendo 39 casas de ambientes de /Users/camiladuelisviana/Desktop/MORE/Integracao/casas_ambientes.xlsx...
Casas sendo processadas: {'R2A28', 'P1A14', 'V1A12', 'P1P15', 'R2P25', 'V1A01', 'I1A41', 'C2P11', 'V1A42', 'R2P34', 'P2P05', 'R1P27', 'C2P10', 'V2P03', 'V1A26', 'R1A04', 'V2P32', 'V2P35', 'R1A06', 'V1P08', 'V1P21', 'R3P39', 'R4P38', 'R1P37', 'R1P09', 'V2P19', 'V1P33', 'V1P20', 'V1P36', 'R1A13', 'V3P24', 'R3A02', 'V1A16', 'R3P18', 'V1A17', 'V1A40', 'P2P29', 'R1A07', 'V1P31'}
DEBUG 1: O índice de 'dados' (antes do merge) é único? True
DEBUG 2: O índice de 'out' (APÓS o merge) é único? True
DEBUG 3: O índice de 'out' (APÓS o reset_index) é único? True
DEBUG 4: Prester a executar o '.loc' que pode causar erro...
DEBUG 5: '.loc' executado com sucesso.
DEBUG 6: Prester a executar o 'np.select' que causa o erro...
DEBUG 7: 'np.select' executado com su